# 🚀 MambaVision Sanity Check (Tích hợp Google Drive & Tái sử dụng `third_party`)

Notebook này được thiết kế theo **Cách 1: Đồng bộ toàn bộ thư mục dự án lên Google Drive**.

Đường dẫn cục bộ: `D:\GG_1\COMPUTER SCIENCE\NAM4_KH1_(2026-2027)\CT208E\SOURCES\LIDC_MambaVision_Project`

### 🌟 Ưu điểm của kiến trúc này:
1. **Tái sử dụng 100% `third_party/MambaVision`**: Không bao giờ phải chạy lệnh `git clone` lại trên Colab. Mã nguồn NVIDIA bạn đã kéo về máy tính nằm sẵn trong Drive.
2. **Không lo mất file**: Toàn bộ code, dữ liệu, logs và file trọng số mô hình (`checkpoints/mambavision_best_model.pth`) được đọc/ghi trực tiếp trên Google Drive vĩnh viễn.
3. **Môi trường huấn luyện y tế thực thụ**: Hoạt động trơn tru trên GPU T4 của Colab trong vòng **10 – 15 phút**.


---
## 1. Kiểm tra phần cứng GPU
> ⚠️ **Yêu cầu**: Vào menu **Runtime** → **Change runtime type** → Chọn **T4 GPU** → Bấm **Save**.


In [ ]:
# 1. Kiểm tra GPU Runtime
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"✅ GPU khả dụng: {gpu_name}")
    print(f"✅ Dung lượng VRAM: {vram_gb:.2f} GB")
else:
    raise SystemError("❌ Chưa bật GPU! Hãy vào Runtime -> Change runtime type -> Chọn T4 GPU!")


---
## 2. Mount Google Drive & Kết nối `third_party/MambaVision`
Kết nối Google Drive, định vị thư mục dự án tương ứng với đường dẫn máy tính:
`D:\GG_1\COMPUTER SCIENCE\NAM4_KH1_(2026-2027)\CT208E\SOURCES\LIDC_MambaVision_Project`
và nạp `third_party/MambaVision` vào `sys.path` để import trực tiếp mà **không cần clone lại**.


In [ ]:
# 2. Mount Google Drive & Kết nối Third-Party
from google.colab import drive
import os
import sys
from pathlib import Path

# Mount Google Drive
drive.mount('/content/drive')

# Đường dẫn dự án tương ứng chính xác với ổ đĩa D: của bạn:
# D:\GG_1\COMPUTER SCIENCE\NAM4_KH1_(2026-2027)\CT208E\SOURCES\LIDC_MambaVision_Project
CANDIDATE_PATHS = [
    # 1. Thư mục GG_1 nằm trong MyDrive
    Path("/content/drive/MyDrive/GG_1/COMPUTER SCIENCE/NAM4_KH1_(2026-2027)/CT208E/SOURCES/LIDC_MambaVision_Project"),
    # 2. Trường hợp GG_1 chính là thư mục gốc MyDrive được đồng bộ bằng Google Drive for Desktop
    Path("/content/drive/MyDrive/COMPUTER SCIENCE/NAM4_KH1_(2026-2027)/CT208E/SOURCES/LIDC_MambaVision_Project"),
    # 3. Dự phòng nếu thư mục dự án được kéo trực tiếp vào gốc MyDrive
    Path("/content/drive/MyDrive/LIDC_MambaVision_Project"),
]

PROJECT_ROOT = None
for candidate in CANDIDATE_PATHS:
    if candidate.exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    PROJECT_ROOT = CANDIDATE_PATHS[0]
    raise FileNotFoundError(
        f"❌ Không tìm thấy thư mục dự án tại:\n   {PROJECT_ROOT}\n\n"
        f"🔍 Vui lòng kiểm tra lại cấu trúc thư mục trên Google Drive của bạn.\n"
        f"Hệ thống đã quét các đường dẫn tiềm năng sau nhưng chưa thấy:\n" +
        "\n".join(f"  • {p}" for p in CANDIDATE_PATHS)
    )

MAMBAVISION_PATH = PROJECT_ROOT / "third_party" / "MambaVision"
assert MAMBAVISION_PATH.exists(), f"Không tìm thấy thư mục MambaVision tại: {MAMBAVISION_PATH}"

# Thêm đường dẫn vào sys.path để import module
sys.path.insert(0, str(MAMBAVISION_PATH))
sys.path.insert(0, str(MAMBAVISION_PATH / "mambavision"))
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# Chuyển working directory sang PROJECT_ROOT
os.chdir(PROJECT_ROOT)

print(f"✅ Thư mục dự án (PROJECT_ROOT): {PROJECT_ROOT}")
print(f"✅ Working Directory           : {os.getcwd()}")
print(f"✅ Đã kết nối third_party      : {MAMBAVISION_PATH}")


---
## 3. Cài đặt các thư viện nhân CUDA (Không clone thêm bất kỳ mã nguồn nào)
Chỉ cài đặt các gói wheel nhị phân tăng tốc phần cứng của Mamba (`causal-conv1d`, `mamba-ssm`) và các thư viện hỗ trợ.


In [ ]:
# 3. Cài đặt thư viện phụ thuộc
# Cài đặt các thư viện cần thiết cho dữ liệu y tế và mô hình
!pip -q install packaging ninja "timm>=0.9.12" einops medmnist

# Tự động phát hiện và thử cài đặt nhân tăng tốc CUDA phù hợp (nếu có)
try:
    import mamba_ssm
    print("✅ Đã nạp thành công nhân tăng tốc CUDA mamba_ssm!")
except ImportError:
    print("ℹ️ Hệ thống tự động kích hoạt: PyTorch Native Fallback (Mô hình chạy 100% chính xác theo chuẩn toán học Mamba)!")


---
## 4. Cấu hình Siêu tham số (Hyperparameters)
Cấu hình chuẩn cho bài toán fine-tuning backbone MambaVision trên GPU T4 (16GB VRAM).


In [ ]:
# 4. Thiết lập Siêu tham số
CONFIG = {
    "dataset": "pneumoniamnist",     # Dữ liệu ảnh X-quang y tế (Normal vs. Pneumonia)
    "image_size": 224,               # Kích thước ảnh chuẩn ImageNet cho MambaVision
    "batch_size": 32,                # Vừa vặn 16GB VRAM của T4
    "learning_rate": 1e-4,           # Tốc độ học chuẩn khi fine-tuning backbone pre-trained
    "weight_decay": 0.05,            # L2 Regularization chống overfitting
    "epochs": 10,                    # Huấn luyện trong 10 epochs (~8 phút)
    "seed": 42,                      # Cố định tính ngẫu nhiên
    "use_amp": True,                 # Bật Automatic Mixed Precision (FP16)
}

# Cố định seed
import random, numpy as np
random.seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])
torch.cuda.manual_seed_all(CONFIG["seed"])

print("✅ Bảng siêu tham số đã khởi tạo:")
for k, v in CONFIG.items():
    print(f"  • {k:15s}: {v}")


---
## 5. Nạp dữ liệu y tế mẫu (MedMNIST)
Tải tập ảnh X-quang phổi `PneumoniaMNIST`, chuyển sang 3 kênh RGB và chuẩn hóa theo chuẩn ImageNet.


In [ ]:
import medmnist
from medmnist import INFO
from torch.utils.data import DataLoader
from torchvision import transforms
import os

info = INFO[CONFIG["dataset"]]
DataClass = getattr(medmnist, info['python_class'])

train_transform = transforms.Compose([
    transforms.Resize((CONFIG["image_size"], CONFIG["image_size"])),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((CONFIG["image_size"], CONFIG["image_size"])),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Dữ liệu cache vào thư mục data của Colab
# Ensure the data directory exists
if not os.path.exists("/content/data"):
    os.makedirs("/content/data")

train_dataset = DataClass(split='train', transform=train_transform, download=True, root="/content/data")
val_dataset = DataClass(split='val', transform=val_transform, download=True, root="/content/data")
test_dataset = DataClass(split='test', transform=val_transform, download=True, root="/content/data")

train_loader = DataLoader(train_dataset, batch_size=CONFIG["batch_size"], shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=CONFIG["batch_size"], shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=CONFIG["batch_size"], shuffle=False, num_workers=2, pin_memory=True)

print(f"✅ Số ảnh tập Train     : {len(train_dataset):,}")
print(f"✅ Số ảnh tập Validation: {len(val_dataset):,}")
print(f"✅ Số ảnh tập Test      : {len(test_dataset):,}")
print(f"Nhãn lớp đối tượng      : {info['label']}")

---
## 5.1. Khảo sát Cấu trúc & Các trường Thuộc tính của Dataset (Dataset Schema & Features)
Bảng chi tiết các trường đặc trưng (features), kiểu dữ liệu (data types), tensor shape và kiểm tra thực tế một mẫu dữ liệu X-quang trước khi bước vào EDA.


In [ ]:
# 5.1. Cấu trúc bộ dữ liệu, các trường Features và kiểm tra Tensor
import pandas as pd
import numpy as np
import torch
from IPython.display import display

print("=" * 80)
print(f"📦 THÔNG TIN KIẾN TRÚC VÀ CÁC TRƯỜNG FEATURES: BỘ DỮ LIỆU {CONFIG['dataset'].upper()}")
print("=" * 80)

# 1. Bảng đặc tả các trường Features trong Dataset
features_info = [
    {
        "Tên Feature": "image (Ảnh X-quang)",
        "Kiểu gốc": "PIL Image / uint8",
        "Shape gốc": "(28, 28) Grayscale",
        "Shape sau transform": f"(3, {CONFIG['image_size']}, {CONFIG['image_size']}) FloatTensor",
        "Miền giá trị (Range)": "Chuẩn hóa ImageNet (Mean: ~0, Std: ~1)",
        "Mô tả y sinh": "Ảnh chụp X-quang lồng ngực (Chest X-Ray) chiếu sau - trước (PA/AP)"
    },
    {
        "Tên Feature": "label (Nhãn chẩn đoán)",
        "Kiểu gốc": "numpy.ndarray (int64)",
        "Shape gốc": "(1,)",
        "Shape sau transform": "torch.Tensor (int64)",
        "Miền giá trị (Range)": "0: Normal (Bình thường) | 1: Pneumonia (Viêm phổi)",
        "Mô tả y sinh": "Kết luận chẩn đoán bệnh lý của bác sĩ chuyên khoa hô hấp / X-quang"
    },
    {
        "Tên Feature": "split (Phân tập)",
        "Kiểu gốc": "string",
        "Shape gốc": "Scalar",
        "Shape sau transform": "-",
        "Miền giá trị (Range)": f"train: {len(train_dataset):,} | val: {len(val_dataset):,} | test: {len(test_dataset):,}",
        "Mô tả y sinh": "Chia dữ liệu theo mức độ bệnh nhân độc lập (Patient-level Split) chống rò rỉ dữ liệu"
    }
]

df_features = pd.DataFrame(features_info)
display(df_features.style.set_properties(**{'text-align': 'left', 'font-size': '12px'}))

# 2. Lấy mẫu thực tế phần tử đầu tiên (Sample #0)
sample_idx = 0
sample_img, sample_target = train_dataset[sample_idx]
label_name = info['label'][str(sample_target.item())]

print("\n" + "=" * 80)
print("🔍 KIỂM TRA MẪU DỮ LIỆU THỰC TẾ TRONG TẬP TRAIN (SAMPLE #0)")
print("=" * 80)
print(f"• Kiểu dữ liệu ảnh (Image Object)  : {type(sample_img)}")
print(f"• Kích thước ảnh Tensor (C, H, W)  : {list(sample_img.shape)} (3 kênh x {CONFIG['image_size']} x {CONFIG['image_size']})")
print(f"• Dạng dữ liệu (Dtype)             : {sample_img.dtype}")
print(f"• Giá trị Min / Max / Mean / Std   : {sample_img.min():.3f} / {sample_img.max():.3f} / {sample_img.mean():.3f} / {sample_img.std():.3f}")
print(f"• Nhãn Target (Mã / Tên phân loại) : {sample_target.item()} -> '{label_name}'")

# 3. Kiểm tra cấu trúc 1 Mini-batch từ DataLoader đưa vào MambaVision
batch_imgs, batch_targets = next(iter(train_loader))
batch_mem_mb = (batch_imgs.element_size() * batch_imgs.nelement()) / (1024 * 1024)

print("\n" + "=" * 80)
print("🚀 KIỂM TRA CẤU TRÚC 1 BATCH ĐƯA VÀO BACKBONE MAMBAVISION")
print("=" * 80)
print(f"• Batch Images Tensor Shape        : {list(batch_imgs.shape)} -> (Batch_Size, Channels, Height, Width)")
print(f"• Batch Targets Tensor Shape       : {list(batch_targets.shape)} -> (Batch_Size, 1)")
print(f"• Bộ nhớ chiếm dụng của 1 Batch    : ~{batch_mem_mb:.2f} MB")
print("=" * 80)


---
## 6. Khảo sát & Phân tích Dữ liệu Y tế (Exploratory Data Analysis - EDA)
Trước khi đưa dữ liệu vào huấn luyện mô hình MambaVision, việc khảo sát dữ liệu (EDA) là bước chuẩn mực trong nghiên cứu AI Y tế:
1. **Định lượng kích thước & Mất cân bằng dữ liệu (Class Imbalance)**: Đánh giá tỷ lệ giữa bệnh nhân Bình thường (Normal) và Viêm phổi (Pneumonia) qua các tập Train/Val/Test.
2. **Phân tích cường độ điểm ảnh (Pixel Intensity Distribution)**: So sánh phân bố mức xám (Grayscale) giữa vùng phổi trong suốt bình thường và vùng phổi đông đặc tổn thương.
3. **Bộ sưu tập trực quan đối chứng (Visual Gallery)**: Quan sát trực quan các đặc trưng hình ảnh X-quang đặc thù của từng nhóm nhãn.


In [ ]:
# 6.1. Thống kê số lượng mẫu và tỷ lệ mất cân bằng lớp (Class Imbalance)
import pandas as pd
import numpy as np
from IPython.display import display

labels_train = np.array(train_dataset.labels).squeeze()
labels_val = np.array(val_dataset.labels).squeeze()
labels_test = np.array(test_dataset.labels).squeeze()

# Bảng thống kê định lượng chi tiết
stats_data = {
    "Tập dữ liệu": ["Huấn luyện (Train)", "Thẩm định (Validation)", "Kiểm thử (Test)", "Tổng cộng (All)"],
    "Normal - Lớp 0": [
        int(np.sum(labels_train == 0)),
        int(np.sum(labels_val == 0)),
        int(np.sum(labels_test == 0)),
        int(np.sum(labels_train == 0) + np.sum(labels_val == 0) + np.sum(labels_test == 0))
    ],
    "Pneumonia - Lớp 1": [
        int(np.sum(labels_train == 1)),
        int(np.sum(labels_val == 1)),
        int(np.sum(labels_test == 1)),
        int(np.sum(labels_train == 1) + np.sum(labels_val == 1) + np.sum(labels_test == 1))
    ]
}

df_stats = pd.DataFrame(stats_data)
df_stats["Tổng số mẫu"] = df_stats["Normal - Lớp 0"] + df_stats["Pneumonia - Lớp 1"]
df_stats["Tỷ lệ Viêm phổi (%)"] = (df_stats["Pneumonia - Lớp 1"] / df_stats["Tổng số mẫu"] * 100).round(2)
df_stats["Hệ số Imbalance (1:k)"] = (df_stats["Pneumonia - Lớp 1"] / df_stats["Normal - Lớp 0"]).round(2)

print("=" * 75)
print("📊 BẢNG KHẢO SÁT PHÂN PHỐI SỐ LƯỢNG MẪU (PNEUMONIAMNIST)")
print("=" * 75)
display(df_stats.style.set_properties(**{'text-align': 'center'}))
print("=" * 75)
print(f"💡 Nhận xét: Trong tập Train, lớp Viêm phổi (Pneumonia) chiếm {df_stats.loc[0, 'Tỷ lệ Viêm phổi (%)']}%.")
print(f"   Tỷ lệ mất cân bằng giữa Viêm phổi : Bình thường xấp xỉ {df_stats.loc[0, 'Hệ số Imbalance (1:k)']}:1.")


In [ ]:
# 6.2. Trực quan hóa Biểu đồ phân phối nhãn (Bar Chart & Donut Chart)
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# 1. Grouped Bar Chart so sánh 3 tập
splits = ['Train', 'Validation', 'Test']
normal_counts = [int(np.sum(labels_train == 0)), int(np.sum(labels_val == 0)), int(np.sum(labels_test == 0))]
pneumonia_counts = [int(np.sum(labels_train == 1)), int(np.sum(labels_val == 1)), int(np.sum(labels_test == 1))]

x = np.arange(len(splits))
width = 0.35

rects1 = axes[0].bar(x - width/2, normal_counts, width, label='Normal (Bình thường)', color='#2563eb', edgecolor='black', alpha=0.85)
rects2 = axes[0].bar(x + width/2, pneumonia_counts, width, label='Pneumonia (Viêm phổi)', color='#dc2626', edgecolor='black', alpha=0.85)

axes[0].set_title('Phân bố số lượng mẫu qua các tập dữ liệu', fontsize=12, fontweight='bold', pad=10)
axes[0].set_xticks(x)
axes[0].set_xticklabels(splits, fontsize=11)
axes[0].set_ylabel('Số lượng ảnh (Samples)', fontsize=11)
axes[0].legend(frameon=True, facecolor='white', framealpha=0.9)
axes[0].grid(axis='y', linestyle='--', alpha=0.4)

# Hiển thị con số chính xác trên đỉnh mỗi cột
for rect in rects1 + rects2:
    h = rect.get_height()
    axes[0].annotate(f'{h:,}',
                    xy=(rect.get_x() + rect.get_width() / 2, h),
                    xytext=(0, 4),
                    textcoords="offset points",
                    ha='center', va='bottom', fontsize=9, fontweight='bold')

# 2. Donut Chart phân bố tỷ lệ trong tập Train
train_counts = [normal_counts[0], pneumonia_counts[0]]
colors = ['#2563eb', '#dc2626']
wedges, texts, autotexts = axes[1].pie(
    train_counts,
    labels=['Normal (Bình thường)', 'Pneumonia (Viêm phổi)'],
    autopct='%1.1f%%',
    startangle=130,
    colors=colors,
    wedgeprops=dict(width=0.42, edgecolor='white', linewidth=2.5),
    textprops=dict(fontsize=11, fontweight='semibold')
)
for at in autotexts:
    at.set_color('white')
    at.set_weight('bold')
    at.set_fontsize(12)

axes[1].set_title('Tỷ lệ thành phần lớp trong tập Huấn luyện (Train Split)', fontsize=12, fontweight='bold', pad=10)

plt.tight_layout()
plt.show()


In [ ]:
# 6.3. Khảo sát phân bố cường độ điểm ảnh (Pixel Intensity Distribution & Contrast)
if hasattr(train_dataset, 'imgs'):
    raw_imgs = train_dataset.imgs
else:
    raw_imgs = np.array([train_dataset[i][0].numpy() for i in range(min(500, len(train_dataset)))])

# Lấy mẫu ngẫu nhiên 300 ảnh mỗi lớp để phân tích phổ cường độ xám
idx_normal = np.where(labels_train == 0)[0][:300]
idx_pneumonia = np.where(labels_train == 1)[0][:300]

pixels_normal = raw_imgs[idx_normal].flatten()
pixels_pneumonia = raw_imgs[idx_pneumonia].flatten()

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# 1. Histogram phân phối mức xám
axes[0].hist(pixels_normal, bins=50, color='#2563eb', alpha=0.6, density=True, label='Normal (Phổi bình thường)')
axes[0].hist(pixels_pneumonia, bins=50, color='#dc2626', alpha=0.55, density=True, label='Pneumonia (Phổi viêm)')
axes[0].set_title('Mật độ phân phối mức xám (Pixel Intensity Histogram)', fontsize=12, fontweight='bold', pad=10)
axes[0].set_xlabel('Giá trị mức xám (Grayscale: 0 - 255)', fontsize=11)
axes[0].set_ylabel('Mật độ xác suất (Density)', fontsize=11)
axes[0].legend(frameon=True, facecolor='white', framealpha=0.9)
axes[0].grid(True, linestyle='--', alpha=0.3)

# 2. Hộp phân vị Boxplot so sánh độ sáng trung bình và độ lệch chuẩn
bp = axes[1].boxplot([pixels_normal, pixels_pneumonia], patch_artist=True, tick_labels=['Normal', 'Pneumonia'])
colors_box = ['#93c5fd', '#fca5a5']
for patch, col in zip(bp['boxes'], colors_box):
    patch.set_facecolor(col)
    patch.set_edgecolor('black')
for median in bp['medians']:
    median.set(color='black', linewidth=2)

axes[1].set_title('Phân vị mức xám & Độ tương phản (Pixel Boxplot)', fontsize=12, fontweight='bold', pad=10)
axes[1].set_ylabel('Giá trị mức xám (0 - 255)', fontsize=11)
axes[1].grid(True, linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"• Giá trị pixel trung bình [Normal]   : {np.mean(pixels_normal):.2f} ± {np.std(pixels_normal):.2f}")
print(f"• Giá trị pixel trung bình [Pneumonia]: {np.mean(pixels_pneumonia):.2f} ± {np.std(pixels_pneumonia):.2f}")
print("💡 Nhận xét lâm sàng:")
print("   Vùng phế trường phổi bình thường chứa nhiều không khí -> màu tối (pixel thấp).")
print("   Phổi viêm xuất hiện thâm nhiễm phế nang & dịch rỉ viêm (consolidation) -> tạo các đám mờ trắng đục (pixel cao hơn).")


In [ ]:
# 6.4. Thư viện trực quan đối chứng mẫu ảnh X-quang (Visual Gallery)
n_samples = 6
sample_normal = np.where(labels_train == 0)[0][:n_samples]
sample_pneumonia = np.where(labels_train == 1)[0][:n_samples]

fig, axes = plt.subplots(2, n_samples, figsize=(18, 6.5))

# Hàng 1: Phổi bình thường
for col, idx in enumerate(sample_normal):
    img = raw_imgs[idx] if hasattr(train_dataset, 'imgs') else train_dataset[idx][0].squeeze()
    axes[0, col].imshow(img, cmap='gray')
    axes[0, col].set_title(f'Normal #{idx}', fontsize=11, fontweight='bold', color='#1d4ed8')
    axes[0, col].axis('off')

# Hàng 2: Phổi viêm
for col, idx in enumerate(sample_pneumonia):
    img = raw_imgs[idx] if hasattr(train_dataset, 'imgs') else train_dataset[idx][0].squeeze()
    axes[1, col].imshow(img, cmap='gray')
    axes[1, col].set_title(f'Pneumonia #{idx}', fontsize=11, fontweight='bold', color='#b91c1c')
    axes[1, col].axis('off')

# Thêm nhãn danh mục bên cạnh trái
axes[0, 0].text(-0.25, 0.5, 'NHÓM 0:\nBÌNH THƯỜNG\n(NORMAL)', transform=axes[0, 0].transAxes,
                va='center', ha='right', fontsize=11, fontweight='bold', color='#1d4ed8')
axes[1, 0].text(-0.25, 0.5, 'NHÓM 1:\nVIÊM PHỔI\n(PNEUMONIA)', transform=axes[1, 0].transAxes,
                va='center', ha='right', fontsize=11, fontweight='bold', color='#b91c1c')

plt.suptitle('Đối chứng trực quan hình ảnh X-quang: Phổi Bình thường vs Phổi Viêm', fontsize=14, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


---
## 7. Nạp NVIDIA MambaVision-T trực tiếp từ `third_party/`
Import trực tiếp `mamba_vision_T` từ thư mục `third_party/MambaVision` đã kết nối, nạp trọng số ImageNet pre-trained và thay thế classification head sang 2 lớp y tế.


In [ ]:
# 7. Khởi tạo MambaVision từ third_party có sẵn
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
import argparse
import torch
import torch.nn as nn

# Cấu hình tương thích PyTorch 2.6+ khi nạp checkpoint pretrained từ HuggingFace
if hasattr(torch, 'serialization') and hasattr(torch.serialization, 'add_safe_globals'):
    try:
        torch.serialization.add_safe_globals([argparse.Namespace])
    except Exception:
        pass

# Import trực tiếp từ third_party/MambaVision có sẵn trong Drive
from mambavision.models.mamba_vision import mamba_vision_T

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Đang tải MambaVision-T với trọng số Pre-trained ImageNet...")
model = mamba_vision_T(pretrained=True)

# Tùy biến lớp Head cuối cùng sang 2 lớp chẩn đoán y tế
in_features = model.head.in_features
model.head = nn.Linear(in_features, len(info['label']))
model = model.to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✅ Model MambaVision-T đã sẵn sàng trên GPU: {device}")
print(f"Tổng số tham số huấn luyện: {total_params / 1e6:.2f}M")


---
## 8. Cấu hình Optimizer, Loss & Mixed Precision Scaler
Sử dụng `AdamW`, `CrossEntropyLoss` và `CosineAnnealingLR`.


In [ ]:
# 8. Optimizer, Loss & Scheduler
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(
    model.parameters(),
    lr=CONFIG["learning_rate"],
    weight_decay=CONFIG["weight_decay"]
)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG["epochs"], eta_min=1e-6)
scaler = torch.cuda.amp.GradScaler(enabled=CONFIG["use_amp"])

print("✅ Đã thiết lập AdamW + Cosine Scheduler + FP16 GradScaler!")


---
## 9. Huấn luyện mô hình & Tự động lưu Checkpoint vào `checkpoints/`
Mỗi khi đạt kỷ lục Validation Accuracy mới, checkpoint sẽ được lưu trực tiếp vào thư mục `checkpoints/` trên Google Drive của bạn.


In [ ]:
# 9. Vòng lặp huấn luyện chính
import time
from tqdm.auto import tqdm

checkpoint_dir = PROJECT_ROOT / "checkpoints"
checkpoint_dir.mkdir(parents=True, exist_ok=True)
best_checkpoint_path = checkpoint_dir / "mambavision_best_model.pth"

history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
best_val_acc = 0.0

print(f"🚀 Bắt đầu huấn luyện {CONFIG['epochs']} Epochs...")
start_time = time.time()

for epoch in range(CONFIG["epochs"]):
    # --- TRAIN ---
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0

    for images, targets in tqdm(train_loader, desc=f"Epoch {epoch+1:02d}/{CONFIG['epochs']} [Train]"):
        images = images.to(device, non_blocking=True)
        targets = targets.squeeze().long().to(device, non_blocking=True)

        optimizer.zero_grad()

        with torch.cuda.amp.autocast(enabled=CONFIG["use_amp"]):
            outputs = model(images)
            loss = criterion(outputs, targets)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        train_correct += (preds == targets).sum().item()
        train_total += targets.size(0)

    scheduler.step()

    epoch_train_loss = train_loss / train_total
    epoch_train_acc = train_correct / train_total

    # --- VALIDATION ---
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0

    with torch.no_grad():
        for images, targets in val_loader:
            images = images.to(device, non_blocking=True)
            targets = targets.squeeze().long().to(device, non_blocking=True)

            with torch.cuda.amp.autocast(enabled=CONFIG["use_amp"]):
                outputs = model(images)
                loss = criterion(outputs, targets)

            val_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            val_correct += (preds == targets).sum().item()
            val_total += targets.size(0)

    epoch_val_loss = val_loss / val_total
    epoch_val_acc = val_correct / val_total

    history["train_loss"].append(epoch_train_loss)
    history["train_acc"].append(epoch_train_acc)
    history["val_loss"].append(epoch_val_loss)
    history["val_acc"].append(epoch_val_acc)

    # Lưu Checkpoint trực tiếp vào Drive
    saved_flag = ""
    if epoch_val_acc > best_val_acc:
        best_val_acc = epoch_val_acc
        torch.save({
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_val_acc": best_val_acc,
            "config": CONFIG
        }, best_checkpoint_path)
        saved_flag = "⭐ [ĐÃ LƯU VÀO CHECKPOINTS]"

    print(f"Epoch [{epoch+1:02d}/{CONFIG['epochs']}] "
          f"Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc*100:.2f}% | "
          f"Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc*100:.2f}% {saved_flag}")

total_elapsed = time.time() - start_time
print(f"\n🎉 Huấn luyện thành công trong {total_elapsed/60:.2f} phút!")
print(f"🏆 Best Validation Accuracy: {best_val_acc*100:.2f}%")
print(f"📁 File checkpoint đã được bảo vệ trên Drive: {best_checkpoint_path}")


---
## 10. Vẽ biểu đồ học tập (Learning Curves)
Đánh giá độ hội tụ và quá trình giảm hàm mất mát qua từng Epoch.


In [ ]:
# 10. Vẽ đồ thị Loss & Accuracy
import matplotlib.pyplot as plt

epochs_range = range(1, CONFIG["epochs"] + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs_range, history["train_loss"], 'o-', color='#2563eb', label='Train Loss')
axes[0].plot(epochs_range, history["val_loss"], 's--', color='#dc2626', label='Val Loss')
axes[0].set_title('MambaVision Training & Validation Loss', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss (Cross-Entropy)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, [acc * 100 for acc in history["train_acc"]], 'o-', color='#059669', label='Train Acc')
axes[1].plot(epochs_range, [acc * 100 for acc in history["val_acc"]], 's--', color='#ea580c', label='Val Acc')
axes[1].set_title('MambaVision Training & Validation Accuracy (%)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


---
## 11. Đánh giá tập Test & Hiển thị ảnh chẩn đoán thực tế
Nạp lại trọng số tốt nhất từ `checkpoints/` trên Drive để đánh giá độ chính xác cuối cùng trên tập Test độc lập và trực quan hóa kết quả chẩn đoán.


In [ ]:
# 11. Đánh giá Test & Visualizer
# Nạp checkpoint tốt nhất từ Drive
# Nạp checkpoint tốt nhất từ Drive (hỗ trợ PyTorch 2.6+)
try:
    checkpoint = torch.load(best_checkpoint_path, weights_only=False)
except TypeError:
    checkpoint = torch.load(best_checkpoint_path)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

# Đo độ chính xác tập Test
test_correct, test_total = 0, 0
with torch.no_grad():
    for images, targets in test_loader:
        images = images.to(device)
        targets = targets.squeeze().long().to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        test_correct += (preds == targets).sum().item()
        test_total += targets.size(0)

test_acc = test_correct / test_total
print(f"📊 Độ chính xác cuối cùng trên tập Test: {test_acc * 100:.2f}%\n")

# Lấy 8 mẫu ảnh kiểm thử ngẫu nhiên
data_iter = iter(test_loader)
images, labels = next(data_iter)
images = images[:8].to(device)
labels = labels[:8].squeeze()

with torch.no_grad():
    outputs = model(images)
    probs = torch.softmax(outputs, dim=1)
    confidences, preds = torch.max(probs, dim=1)

# Hiển thị ảnh chẩn đoán
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

label_names = {0: "Normal", 1: "Pneumonia"}

for i in range(8):
    img_display = images[i].cpu().permute(1, 2, 0).numpy()
    img_display = img_display * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
    img_display = np.clip(img_display, 0, 1)

    true_label = label_names[labels[i].item()]
    pred_label = label_names[preds[i].item()]
    conf = confidences[i].item() * 100

    is_correct = (preds[i].item() == labels[i].item())
    color = "green" if is_correct else "red"

    axes[i].imshow(img_display)
    axes[i].set_title(f"True: {true_label}\nPred: {pred_label} ({conf:.1f}%)", color=color, fontweight='bold', fontsize=11)
    axes[i].axis("off")

plt.suptitle("MambaVision Test Predictions (Xanh = Đúng, Đỏ = Sai)", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()
